# GI Dump to Mod Coverter
[![Static Badge](https://img.shields.io/badge/Jupyter_Notebook-F37726?style=for-the-badge)](https://jupyter.org/)

<br>

## Requirements
- Python (Version 3.6 or up)

<br>
<br>

## Installation
Choose how to install AGRemap's API

**Option A**: If you want to install through [Pypi](https://pypi.org/project/AnimeGameRemap/), you run the pip install command below


In [ ]:
%pip install -U AnimeGameRemap

In [ ]:
import AnimeGameRemap as AGR

<br>

**Option B**: Alternatively, you can locally import the API from a specific git branch

In [1]:
import sys

# Note: Make sure the path correctly points where the AGRemap's API is located
sys.path.insert(1, r"../../../Anime Game Remap (for all users)/api")

import src.FixRaidenBoss2 as AGR

<br>
<br>

## Initialization
Run the codeblock below to initialize the necessary tools for the conversion process.

In [2]:
import sys
import struct
from enum import Enum
from typing import Dict, Type, Optional, List, Callable, Any


class VBPart(Enum):
    Position = "Position"
    Blend = "Blend"
    Texture = "Texcoord"


class VBElementMetadata(Enum):
    SemanticName = "SemanticName"
    Format = "Format"


class StrClassifiers(Enum):
    VBPartClassification = AGR.AhoCorasickBuilder().build(data = {"POSITION": VBPart.Position,
                                                                  "NORMAL": VBPart.Position,
                                                                  "TANGENT": VBPart.Position,
                                                                  "BLEND": VBPart.Blend,
                                                                  "COLOR": VBPart.Texture,
                                                                  "TEXCOORD": VBPart.Texture})
    
    VBDataTypeSize = AGR.AhoCorasickBuilder().build(data = {"8_": 1, "16_": 2, "32_": 4, "64_": 8, "128_": 16})
    VBDataType = AGR.AhoCorasickBuilder().build(data = {"FLOAT": AGR.BufBaseFloat,
                                                        "SINT": AGR.BufSignedInt,
                                                        "UNORM": AGR.BufUnorm})


class FileEncoder():
    def __init__(self, file: str):
        self.file = file
        self._fileLines = []
        self._fileRead = False

    def read(self):
        with open(self.file, "r", encoding = "utf-8") as f:
            self._fileLines  = f.readlines()

        self._fileRead = True

    def skipHeader(self, fileLineInd: int) -> int:
        fileLinesLen = len(self._fileLines)
        result = fileLineInd

        for i in range(fileLineInd, fileLinesLen):
            line = self._fileLines[i].strip()
            if (line == ""):
                result = i
                break

        return result

    def encode(self, flush: bool = False):
        if (flush or not self._fileRead):
            self.read()


class IbEncoder(FileEncoder):
    def __init__(self, file: str, fixedFile: str):
        super().__init__(file)
        self.fixedFile = fixedFile

    def parseTriangle(self, line: str) -> bytes:
        result = b""
        unsignedIntType = AGR.BufUnSignedInt()

        vertices = line.split(" ")
        for vertex in vertices:
            vertexVal = int(vertex)
            result += unsignedIntType.encode(vertexVal)

        return result

    def encode(self, flush: bool = False):
        super().encode(flush = flush)

        result = bytearray()
        fileLineInd = 0
        fileLinesLen = len(self._fileLines)
        parsedHeader = False

        while (fileLineInd < fileLinesLen):
            if (parsedHeader):
                line = self._fileLines[fileLineInd]
                result += self.parseTriangle(line)
            else:
                fileLineInd = self.skipHeader(fileLineInd)
                parsedHeader = True

            fileLineInd += 1

        with open(self.fixedFile, "wb") as f:
            f.write(result)


class VbEncoder(FileEncoder):
    def __init__(self, file: str, fixedPrefix: str):
        super().__init__(file)
        self.fixedPrefix = fixedPrefix
        self.partDataTypes: Dict[VBPart, AGR.BufDataType] = {}
        self.fixedPartFiles: Dict[VBPart, str] = {}

        for part in VBPart:
            self.partDataTypes[part] = []
            self.fixedPartFiles[part] = f"{fixedPrefix}{part.value}.buf"

    def makeDataType(self, dataType: Type[AGR.BufDataType], size: int) -> Optional[AGR.BufDataType]:
        dataTypeName = f"{dataType.__name__}{size}"
        if (dataType == AGR.BufSignedInt):
            return AGR.BufSignedInt(name = dataTypeName, size = size)
        elif (dataType == AGR.BufUnorm):
            return AGR.BufUnorm(dataTypeName, size)
        elif (dataType == AGR.BufBaseFloat and size == 4):
            return AGR.BufFloat()
        elif (dataType == AGR.BufBaseFloat and size == 2):
            return AGR.BufFloat16()
        
        return None

    def parseHeader(self, fileLineInd: int) -> int:
        fileLinesLen = len(self._fileLines)
        result = fileLineInd

        vbPart = None

        for i in range(fileLineInd, fileLinesLen):
            line = self._fileLines[i].strip()
            if (line == ""):
                result = i
                break

            if (vbPart is None and line.find(VBElementMetadata.SemanticName.value) > -1):
                vbPartKeyword, vbPart = StrClassifiers.VBPartClassification.value.getMaximal(line, errorOnNotFound = False)

            elif (vbPart is not None and line.find(VBElementMetadata.Format.value) > -1):
                sizeKeyword, sizeKeywordStartInd = StrClassifiers.VBDataTypeSize.value.findMaximal(line)
                if (sizeKeyword is None):
                    continue
                
                sizeKeywordEndInd = sizeKeywordStartInd + len(sizeKeyword)
                size = StrClassifiers.VBDataTypeSize.value.getKeyVal(sizeKeyword)
                line = line[sizeKeywordEndInd:]

                dataTypeKeyword, dataTypeCls = StrClassifiers.VBDataType.value.getMaximal(line, errorOnNotFound = False)
                if (dataTypeCls is None):
                    continue

                dataType = self.makeDataType(dataTypeCls, size)
                if (dataType is not None):
                    self.partDataTypes[vbPart].append(dataType)

                vbPart = None

        return result
    
    def getDataTypeStrParser(self, dataType: AGR.BufDataType) -> Optional[Callable[[str], Any]]:
        if (isinstance(dataType, AGR.BufSignedInt)):
            return lambda txt: int(txt)
        elif (isinstance(dataType, AGR.BufBaseFloat) or isinstance(dataType, AGR.BufUnorm)):
            return lambda txt: float(txt)
        
        return None
    
    def parseVertex(self, vertexLines: List[str], result: Dict[VBPart, bytearray]):
        vertexLinesLen = len(vertexLines)
        lineInd = 0

        for part in self.partDataTypes:
            if (part not in result):
                continue

            partTypes = self.partDataTypes[part]
            for dataType in partTypes:
                if (lineInd >= vertexLinesLen):
                    return
                
                line = vertexLines[lineInd]
                strParse = self.getDataTypeStrParser(dataType)
                strDataParts = line.split(",")

                for strDataPart in strDataParts:
                    dataPart = strParse(strDataPart)
                    dataPart = dataType.encode(dataPart)
                    result[part] += dataPart

                lineInd += 1


    def encode(self, flush: bool = False):
        super().encode(flush = flush)

        result = {}
        for part in VBPart:
            result[part] = bytearray()

        fileLineInd = 0
        fileLinesLen = len(self._fileLines)
        parsedHeader = False
        vertexLines = []

        while (fileLineInd < fileLinesLen):
            if (not parsedHeader):
                fileLineInd = self.parseHeader(fileLineInd)
                parsedHeader = True
                fileLineInd += 3
                continue

            line = self._fileLines[fileLineInd].strip()
            if (line == ""):
                self.parseVertex(vertexLines, result)
                vertexLines = []
            else:
                lineDataStartPos = line.rfind(":")
                if (lineDataStartPos > -1):
                    line = line[lineDataStartPos + 1:]
                vertexLines.append(line)

            fileLineInd += 1

        if (vertexLines):
            self.parseVertex(vertexLines, result)

        for part in VBPart:
            fixedFile = self.fixedPartFiles[part]
            with open(fixedFile, "wb") as f:
                f.write(result[part])


<br>
<br>

## File Setup
Ensure the file paths are set correctly for the following constants:

- **IBPaths**
- **VBPaths**

In [10]:
import os
import glob
import re


#####################
# Ensure the pathts set in these constants are correct

IBPaths = {}
VBPaths = {}

#####################


ModFolders = {
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Amber": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Amber\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\AmberCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\AmberCN\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Ayaka": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Ayaka\4_0",
    r"C:\Users\3dark\Downloads\AyakaSpringBloom\AyakaSpringBloom": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\AyakaSpringBloom\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Barbara": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Barbara\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\BarbaraSummertime": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\BarbaraSummertime\4_0",
    r"E:\Computer\Downloads\CherryHuTao\CherryHuTao": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\CherryHuTao\5_3",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Diluc": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Diluc\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\DilucFlamme": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\DilucFlamme\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Fischl": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Fischl\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\FischlHighness": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\FischlHighness\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Ganyu": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Ganyu\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\GanyuTwilight": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\GanyuTwilight\4_4",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\HuTao": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\HuTao\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Jean": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Jean\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\JeanCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\JeanCN\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\JeanSea": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\JeanSea\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Keqing": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Keqing\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KeqingOpulent": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KeqingOpulent\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Kirara": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Kirara\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KiraraBoots": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KiraraBoots\4_8",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Klee": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Klee\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\KleeBlossomingStarlight": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\KleeBlossomingStarlight\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Lisa": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Lisa\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\LisaStudent": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\LisaStudent\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Mona": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Mona\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\MonaCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\MonaCN\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Nilou": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Nilou\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\NilouBreeze": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\NilouBreeze\4_8",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Ningguang": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Ningguang\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\NingguangOrchid": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\NingguangOrchid\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Rosaria": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Rosaria\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\RosariaCN": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\RosariaCN\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Shenhe": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Shenhe\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\ShenheFrostFlower": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\ShenheFrostFlower\4_4",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Xiangling": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Xiangling\4_0",
    r"E:\Computer\Downloads\XianglingCheer-corrected\XianglingCheer": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\XianglingCheer\5_3",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Xingqiu": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Xingqiu\4_0",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\XingqiuBamboo": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\XingqiuBamboo\4_4",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Arlecchino": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Arlecchino\4_6",
    r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\RaidenShogun": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\RaidenShogun\4_0",

    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\AyakaSpringbloom": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\AyakaSpringbloom\5_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\LisaStudent": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\LisaStudent\5_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\NilouBreeze": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\NilouBreeze\5_4",
    # r"E:\Computer\Games\Genshin\Repos\Repos\GI-Model-Importer-Assets\PlayerCharacterData\Arlecchino": r"E:\Computer\Games\Genshin\Repos\Repos\Fix-Raiden-Boss\Data\Mod Downloads\GI\Arlecchino\5_4",
}

for srcFolder in ModFolders:
    dstFolder = ModFolders[srcFolder]
    ibPaths = glob.glob(os.path.join(srcFolder, "*-ib*.txt"))
    vbPaths = glob.glob(os.path.join(srcFolder, "*-vb0*.txt"))

    for ibPath in ibPaths:
        ibFileName = os.path.basename(ibPath)
        ibFileName = re.sub(r"-ib.*", ".ib", ibFileName)
        dstIbPath = os.path.join(dstFolder, ibFileName)
        IBPaths[ibPath] = dstIbPath

    if (not vbPaths):
        continue

    vbPath = vbPaths[0]
    vbFilePrefix = os.path.basename(vbPath)
    vbFilePrefix = re.sub(r"-vb0.*", "", vbFilePrefix)
    vbFilePrefix = re.sub(r"Head|Body|Dress|Extra", "", vbFilePrefix)
    VBPaths[vbPath] = os.path.join(dstFolder, vbFilePrefix)

<br>
<br>

## Run the Converter
The code block below converts the dump files into their corresponding binary formats

In [11]:
for srcPath in IBPaths:
    dstPath = IBPaths[srcPath]
    ibEncoder = IbEncoder(srcPath, dstPath)
    ibEncoder.encode()

for srcPath in VBPaths:
    filePrefix = VBPaths[srcPath]
    vbEncoder = VbEncoder(srcPath, filePrefix)
    vbEncoder.encode()